# 23. 네 모델 공동 Test

22번 LR·SVM, 13B CNN, 17B MERT의 설정과 Validation 임계값을 고정한 뒤 같은 Test에서 Baseline·Optimized를 비교했다. Test 점수로 모델이나 임계값을 바꾸지 않았다. 과거 09·13·17번 실행에서 이 Test가 이미 사용된 이력은 별도로 기록한다.

## 1. 선택 산출물 동결 검사

모든 checkpoint, Validation의 Segment/Track 임계값, run ID, manifest hash, CNN cache index와 MERT revision을 확인한 뒤 Test를 연다. 공통 지표는 ROC-AUC, FAKE/REAL AP, 보간 EER, Balanced Accuracy, Macro-F1, REAL FPR, FAKE Miss Rate, confusion matrix와 HTER다. AP는 average_precision_score이며 사다리꼴 PR-AUC와 다르다. Test HTER는 저장된 Validation 임계값의 오류율 평균이고 Test EER은 분리력 지표다.

In [1]:
# 네 모델의 checkpoint·Validation 선택·입력 hash를 확인한 뒤 Test 개봉을 허용한다.
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.modeling_final_run import (
    preflight,
    load_fixed_test,
    infer_classical,
    infer_cnn,
    infer_mert,
    finish_test,
)

# Test를 읽기 전에 네 모델의 Baseline/Optimized 선택을 모두 검사한다.
context = preflight(Path.cwd())
print("Final run:", context["final_run_id"])
print("Component runs:", context["run_ids"])
print("Manifest hash:", context["manifest_hash"])

Final run: final_binary_20260919T164939Z
Component runs: {'classical': 'classical_20260919T110356Z', 'cnn': 'cnn_revised_20260919T110440Z', 'mert': 'mert_binary_20260919T110741Z'}
Manifest hash: 9023dcb21bef5d7163c63ea93258195e53c9fde3ecb7fee5920bf6ff94bbc2b3


`preflight()`가 네 모델의 Baseline·Optimized checkpoint, 전체 후보 수(LR 5·SVM 16·CNN 18·MERT 65), Validation 최상위 선택, 임계값 및 manifest 출처를 확인했다. 최종 run ID는 `final_binary_20260919T164939Z`이다. 구성 run ID는 Classical `classical_20260919T110356Z`, CNN `cnn_revised_20260919T110440Z`, MERT `mert_binary_20260919T110741Z`이며 manifest SHA-256은 `9023dcb21bef5d7163c63ea93258195e53c9fde3ecb7fee5920bf6ff94bbc2b3`이다.

네 모델의 설정과 Segment·Track Validation 임계값이 모두 저장된 상태에서 공동 Test 평가를 시작했다. Test 점수는 위 선택 과정에 입력되지 않았다.

기존 2026-09-13 실험에서 같은 Test의 결과가 이미 공개됐으므로 역사상 완전히 미사용인 Test라고 주장하지 않는다. 첫 실행에서는 CNN 추론 중 Jupyter 커널의 네이티브 OpenMP 충돌이 발생했다. PyTorch import 순서만 수정하고 Validation 추론을 재검증한 뒤 공동 Test 단계를 다시 실행했다. 이 복구 과정에서 모델·checkpoint·임계값은 변경하지 않았으며, 첫 실행에서 Test 성능 지표는 계산되지 않았다.

## 2. 고정 Test 입력

기존 original_audio group split을 다시 만들지 않는다. REAL/FAKE mapping, group 중복, Track label과 266-D feature 순서를 검증한다. 모델별 실패 sample을 조용히 제외하지 않는다.

In [2]:
# 동결 검사를 통과한 같은 Test ID에서 handcrafted feature와 manifest를 읽는다.
# 모든 Validation 선택이 동결된 뒤 처음으로 Test를 연다.
test_meta, X_test, manifest = load_fixed_test(context)
class_counts = (
    manifest.groupby(["split", "label"], sort=False)
    .agg(
        original_audio=("original_audio", "nunique"),
        tracks=("track_sample_id", "nunique"),
        segments=("segment_id", "nunique"),
    )
    .reset_index()
)
display(class_counts)
print("Test segments:", len(test_meta), "Tracks:", test_meta.track_sample_id.nunique())
print("Feature columns:", len(X_test.columns))

,split,label,original_audio,tracks,segments
0,train,REAL,207,207,621
1,val,REAL,44,44,132
2,test,REAL,45,45,135
3,train,FAKE,207,2185,6346
4,test,FAKE,45,494,1437
5,val,FAKE,44,483,1406


Test segments: 1572 Tracks: 539
Feature columns: 266


고정 Test에는 45개 `original_audio` group에서 나온 539 Track과 1,572 Segment가 있다. REAL은 45 Track·135 Segment, FAKE는 494 Track·1,437 Segment다. REAL 비율은 Track 8.35%, Segment 8.59%다. Train은 REAL/FAKE Segment 621/6,346, Validation은 132/1,406이며, handcrafted 입력은 저장된 순서의 266개 열이다.

같은 원곡 group의 REAL과 파생 FAKE를 원래 split 안에 유지하고, 모든 모델의 두 변형에 동일한 Test ID를 사용한다. FAKE가 약 91.65%인 Track 집합이므로 class별 오류를 따로 봐야 한다.

FAKE가 다수이므로 높은 FAKE AP나 전체 Accuracy만으로 REAL 오탐이 낮다고 해석할 수 없다. Track 539개 중 REAL은 45개여서 REAL FPR의 한 건 변화가 약 2.22%p다.

## 3. LR와 RBF-SVM Test 추론

Pipeline의 StandardScaler는 Train에서만 fit되었다. LR은 classes_의 FAKE 열 predict_proba, SVM은 probability=False의 decision_function margin을 사용한다. 두 모델에 class_weight=balanced가 적용되었다. 동일 후보의 Baseline과 Optimized는 score를 재사용한다. C는 정규화 강도와 관련되고 SVM gamma는 RBF 곡면의 영향을 조절한다. 이 모델들은 CNN처럼 직접 learning rate, batch size, epoch를 지정하지 않는다.

In [3]:
# Train-only scaler가 들어 있는 LR/SVM Pipeline으로 Test score를 계산한다.
# Train-only scaler와 분류기가 함께 저장된 Pipeline으로 추론한다.
classical_scores, classical_times = infer_classical(context, test_meta, X_test)
display(pd.DataFrame(classical_times))

,model,variant,seconds,reused_from
0,LogisticRegression,baseline,0.001673,None
1,LogisticRegression,optimized,0.001254,None
2,RBF-SVM,baseline,0.288384,None
3,RBF-SVM,optimized,0.196787,None


1,572개 Test Segment의 추론 시간은 LR Baseline 0.0017초·Optimized 0.0013초, RBF-SVM Baseline 0.2884초·Optimized 0.1968초였다. 네 변형 모두 고정 Test ID 전체에 대해 FAKE 방향 score를 생성했다.

LR은 FAKE 열의 `predict_proba`, SVM은 `decision_function` margin을 사용했다. 각 Pipeline의 scaler는 Train에만 적합됐고, 저장된 feature 열 순서로 Test를 변환했다.

이는 한 번의 모델 추론 시간이며 feature 추출, 파일 읽기, 학습 시간은 포함하지 않는다. SVM margin을 확률로 해석하거나 서로 다른 모델의 score 크기를 직접 비교하지 않는다.

## 4. Log-Mel CNN Test 추론

기존 네 convolution block과 Log-Mel 설정을 유지한다. 입력 [1,128,T]의 1은 채널, 128은 Mel bin, T는 시간 frame이다. 기존 24 kHz, n_fft=1024, hop=240, center=True에서는 T=1001이다. 1001은 10초 길이만으로 정해지지 않는다. 128 Mel bin은 128 handcrafted feature가 아니다. CNN sigmoid score를 보정된 실제 확률로 단정하지 않는다.

In [4]:
# Validation best epoch CNN checkpoint를 복원해 같은 Test Segment를 추론한다.
# Validation Track EER로 선택된 best epoch checkpoint를 복원한다.
cnn_scores, cnn_times, cnn_device = infer_cnn(context, test_meta, manifest)
print("CNN device:", cnn_device)
display(pd.DataFrame(cnn_times))

CNN device: mps


,model,variant,seconds,reused_from
0,Log-Mel CNN,baseline,4.880270,None
1,Log-Mel CNN,optimized,4.063502,None


MPS에서 CNN Baseline의 1,572개 Test Segment 추론은 4.8803초, Optimized는 4.0635초였다. Optimized 설정은 학습률 `3e-4`, dropout `0.3`, weight decay `1e-3`, batch size `16`이며 선택 checkpoint는 17 epoch다.

Train에서 선택한 두 checkpoint를 그대로 복원해 같은 Log-Mel cache에서 FAKE 방향 sigmoid score를 계산했다. Optimized와 Baseline은 별도 Validation 임계값을 갖는다.

두 추론 시간은 한 번 측정한 값이며 batch size도 달라 일반적인 속도 차이라고 단정하지 않는다. `pos_weight`를 사용한 sigmoid score를 보정된 실제 발생 확률로 해석하지 않는다.

## 5. Frozen MERT Test embedding과 LR 추론

MERT-v1-95M의 24 kHz processor를 사용한다. hidden_states[0]은 첫 Transformer block 이전, 1–12는 각 block 출력으로 총 13 representation level이다. 각 level은 768-D이며 13×768을 붙여 사용하지 않는다. frozen encoder를 eval/inference mode로 두고 padding 없는 유효 frame 평균을 계산한다. Test cache는 모든 선택이 끝난 이 단계에서 만든다. MERT는 외부 사전학습을 사용한다.

In [5]:
# 고정 revision MERT embedding에서 저장된 layer와 LR head만 사용한다.
# 동일 Test segment의 13개 layer를 한 번 추출하고 선택 layer만 사용한다.
mert_scores, mert_times, mert_details = infer_mert(context, test_meta)
print("MERT extraction:", mert_details)
display(pd.DataFrame(mert_times))

MERT cache: 50 / 1572
MERT cache: 100 / 1572
MERT cache: 150 / 1572
MERT cache: 200 / 1572
MERT cache: 250 / 1572
MERT cache: 300 / 1572
MERT cache: 350 / 1572
MERT cache: 400 / 1572
MERT cache: 450 / 1572
MERT cache: 500 / 1572
MERT cache: 550 / 1572
MERT cache: 600 / 1572
MERT cache: 650 / 1572
MERT cache: 700 / 1572
MERT cache: 750 / 1572
MERT cache: 800 / 1572
MERT cache: 850 / 1572
MERT cache: 900 / 1572
MERT cache: 950 / 1572
MERT cache: 1000 / 1572
MERT cache: 1050 / 1572
MERT cache: 1100 / 1572
MERT cache: 1150 / 1572
MERT cache: 1200 / 1572
MERT cache: 1250 / 1572
MERT cache: 1300 / 1572
MERT cache: 1350 / 1572
MERT cache: 1400 / 1572
MERT cache: 1450 / 1572
MERT cache: 1500 / 1572
MERT cache: 1550 / 1572
MERT cache: 1572 / 1572
MERT Test cache complete: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/mert/mert95m_binary_test_validframe_12af15fef9d0_meta_run.json


MERT extraction: {'device': 'cpu', 'model_load_seconds': 0.7226876670029014, 'embedding_extraction_seconds': 448.2287088340381, 'cache_paths': {'embeddings': '/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/mert/mert95m_binary_test_validframe_12af15fef9d0.npy', 'done': '/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/mert/mert95m_binary_test_validframe_12af15fef9d0_done.npy', 'index': '/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/mert/mert95m_binary_test_validframe_12af15fef9d0_index.csv', 'metadata': '/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/mert/mert95m_binary_test_validframe_12af15fef9d0_meta.json'}, 'cache_metadata': {'model': 'm-a-p/MERT-v1-95M', 'revision': '12af15fef9d0ac838c3f475bfbbf26d2060dd4f5', 'processor_sampling_rate': 24000, 'segment_sec': 10.0, 'input': '24kHz mono, original 10s boundary, no zero padding', 'pooling': 'single-segment valid hidden frames mean', 'l

,model,variant,seconds,reused_from
0,Frozen MERT + LR,baseline,0.005119,None
1,Frozen MERT + LR,optimized,0.008157,None


MERT 모델 로드는 0.723초, CPU에서 1,572개 Test Segment의 13×768 embedding 추출은 448.229초였다. 저장된 LR head 추론은 Baseline 0.0051초, Optimized 0.0082초였다. Baseline은 layer 12·C=1, Optimized는 layer 4·C=0.01이다.

encoder 가중치는 동결됐고 동일 Test embedding cache를 두 LR head에서 재사용했다. 각 head는 저장된 Train 전용 scaler를 거쳐 FAKE 확률을 출력했다.

LR head의 밀리초 단위 시간은 MERT 표현 추출 비용을 제외한다. MERT는 외부 사전학습 정보를 사용하므로 처음부터 학습한 CNN과 학습 정보량이 같지 않다. 사전학습 자료와 본 평가 자료의 중복 여부는 확인하지 못했다.

## 6. 공통 Test 평가와 Baseline 비교

Segment와 Track에 저장된 서로 다른 Validation 임계값을 적용한다. prediction은 score >= threshold다. Test EER을 계산하지만 그 threshold로 Test 분류를 바꾸지 않는다. Test Balanced Accuracy, Macro-F1, REAL FPR, FAKE Miss Rate, confusion matrix 및 HTER에는 Validation 임계값을 사용한다. 네 모델의 Baseline과 Optimized 결과를 모두 보존한다.

In [6]:
# 여덟 score 집합을 공통 평가 함수로 비교하고 Validation 임계값을 그대로 적용한다.
# 정확히 같은 Test ID의 여덟 score를 한 최종 단계에서 평가한다.
all_scores = {**classical_scores, **cnn_scores, **mert_scores}
all_times = classical_times + cnn_times + mert_times
metrics, track_comparison, segment_comparison, optimized = finish_test(
    context, test_meta, all_scores, all_times, cnn_device, mert_details
)
import json

# 선택 정보와 Test 지표를 한 표에 함께 표시한다.
best_hyperparameters = {
    "LogisticRegression": {
        "C": context["classical_best"]["models"]["LogisticRegression"]["params"]["C"]
    },
    "RBF-SVM": {
        key: context["classical_best"]["models"]["RBF-SVM"]["params"][key]
        for key in ("C", "gamma")
    },
    "Log-Mel CNN": {
        key: context["cnn_selection"]["optimized"]["trial"][key]
        for key in (
            "lr",
            "dropout",
            "weight_decay",
            "batch_size",
            "max_epochs",
            "patience",
        )
    },
    "Frozen MERT + LR": {
        key: context["mert_checkpoint"]["optimized"][key] for key in ("layer", "C")
    },
}
track_display = track_comparison.copy()
track_display["best_hyperparameters"] = track_display["model"].map(
    lambda model: json.dumps(best_hyperparameters[model], ensure_ascii=False)
)
display(
    track_display[
        [
            "model",
            "baseline_roc_auc",
            "optimized_roc_auc",
            "delta_roc_auc",
            "baseline_eer",
            "optimized_eer",
            "delta_eer",
            "baseline_balanced_accuracy",
            "optimized_balanced_accuracy",
            "baseline_macro_f1",
            "optimized_macro_f1",
            "best_hyperparameters",
        ]
    ]
)
display(
    optimized.loc[
        optimized.level.eq("track"),
        [
            "model",
            "n_real",
            "n_fake",
            "real_prevalence",
            "fake_prevalence",
            "roc_auc",
            "ap_fake",
            "ap_real",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "hter",
        ],
    ]
)
print("CSV, raw scores and PNG saved to", context["out"])

,model,baseline_roc_auc,optimized_roc_auc,delta_roc_auc,baseline_eer,optimized_eer,delta_eer,baseline_balanced_accuracy,optimized_balanced_accuracy,baseline_macro_f1,optimized_macro_f1,best_hyperparameters
0,Frozen MERT + LR,0.966442,0.984480,0.018039,0.088889,0.066667,-0.022222,0.896851,0.936325,0.751532,0.878657,"{""layer"": 4, ""C"": 0.01}"
1,Log-Mel CNN,0.967296,0.984525,0.017229,0.111111,0.074899,-0.036212,0.902946,0.887854,0.793301,0.876437,"{""lr"": 0.0003, ""dropout"": 0.3, ""weight_decay"":..."
2,LogisticRegression,0.917634,0.941880,0.024247,0.155556,0.133333,-0.022222,0.858480,0.886775,0.744340,0.772574,"{""C"": 0.01}"
3,RBF-SVM,0.971660,0.958614,-0.013045,0.088889,0.125506,0.036617,0.920130,0.899955,0.818467,0.847002,"{""C"": 10.0, ""gamma"": 0.001}"


,model,n_real,n_fake,real_prevalence,fake_prevalence,roc_auc,ap_fake,ap_real,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate,hter
3,LogisticRegression,45,494,0.083488,0.916512,0.941880,0.993689,0.721660,0.133333,0.886775,0.772574,0.133333,0.093117,0.113225
7,RBF-SVM,45,494,0.083488,0.916512,0.958614,0.995611,0.840145,0.125506,0.899955,0.847002,0.155556,0.044534,0.100045
11,Log-Mel CNN,45,494,0.083488,0.916512,0.984525,0.998539,0.901419,0.074899,0.887854,0.876437,0.200000,0.024291,0.112146
15,Frozen MERT + LR,45,494,0.083488,0.916512,0.984480,0.998330,0.937710,0.066667,0.936325,0.878657,0.088889,0.038462,0.063675


CSV, raw scores and PNG saved to /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/results/model_tuning


고정 Test Track은 REAL 45개·FAKE 494개다. Delta는 Optimized−Baseline이며 EER는 낮을수록 좋다.

| 모델 | AUC Baseline→Optimized (Delta) | EER Baseline→Optimized (Delta) | BA Baseline→Optimized | Macro-F1 Baseline→Optimized |
|:--|:--|:--|:--|:--|
| LR | 0.9176→0.9419 (+0.0242) | 0.1556→0.1333 (−0.0222) | 0.8585→0.8868 | 0.7443→0.7726 |
| RBF-SVM | 0.9717→0.9586 (−0.0130) | 0.0889→0.1255 (+0.0366) | 0.9201→0.9000 | 0.8185→0.8470 |
| Log-Mel CNN | 0.9673→0.9845 (+0.0172) | 0.1111→0.0749 (−0.0362) | 0.9029→0.8879 | 0.7933→0.8764 |
| Frozen MERT + LR | 0.9664→0.9845 (+0.0180) | 0.0889→0.0667 (−0.0222) | 0.8969→0.9363 | 0.7515→0.8787 |

**고정 임계값의 보조 지표**

| Optimized 모델 | FAKE AP | REAL AP | REAL FPR | FAKE Miss | Validation 임계값의 Test HTER |
|:--|--:|--:|--:|--:|--:|
| LR | 0.9937 | 0.7217 | 0.1333 | 0.0931 | 0.1132 |
| RBF-SVM | 0.9956 | 0.8401 | 0.1556 | 0.0445 | 0.1000 |
| Log-Mel CNN | 0.9985 | 0.9014 | 0.2000 | 0.0243 | 0.1121 |
| Frozen MERT + LR | 0.9983 | 0.9377 | 0.0889 | 0.0385 | 0.0637 |

별도 **Test Segment** 표는 같은 1,572개 Segment(REAL 135·FAKE 1,437)에 대한 결과다. Track 본표와 단위가 다르므로 수치를 직접 합치지 않는다.

| 모델 | Segment AUC Baseline→Optimized (Delta) | Segment EER Baseline→Optimized (Delta) | Segment BA Baseline→Optimized | Segment Macro-F1 Baseline→Optimized |
|:--|:--|:--|:--|:--|
| LR | 0.8776→0.8935 (+0.0159) | 0.2000→0.1942 (−0.0058) | 0.7946→0.8049 | 0.6633→0.6620 |
| RBF-SVM | 0.9262→0.9134 (−0.0128) | 0.1481→0.1630 (+0.0148) | 0.8459→0.8377 | 0.7107→0.7181 |
| Log-Mel CNN | 0.9425→0.9599 (+0.0174) | 0.1259→0.1185 (−0.0074) | 0.8815→0.8827 | 0.7613→0.7996 |
| Frozen MERT + LR | 0.9461→0.9731 (+0.0270) | 0.1185→0.0786 (−0.0399) | 0.8766→0.9217 | 0.7277→0.8164 |

![네 모델의 Baseline·Optimized Track ROC-AUC 비교](../results/model_tuning/baseline_vs_optimized_auc.png)

![네 모델의 Baseline·Optimized Track EER 비교](../results/model_tuning/baseline_vs_optimized_eer.png)

MERT Optimized는 EER 0.0667, Balanced Accuracy 0.9363, REAL FPR 0.0889, HTER 0.0637로 네 모델 중 균형 잡힌 결과를 보였다. CNN은 AUC 0.984525로 MERT의 0.984480보다 0.000045 높고 FAKE Miss 0.0243으로 가장 낮지만 REAL FPR 0.2000이다. SVM은 Validation 기준으로 선택한 설정이 Test AUC·EER에서는 Baseline보다 나빠졌다. 이 결과도 비교에 그대로 남겼다.

CNN의 EER는 개선됐지만 저장된 Validation 임계값에서 REAL FPR은 0.1111→0.2000, HTER는 0.0971→0.1121로 악화됐다. SVM Macro-F1이 좋아진 것만으로 AUC·EER 악화를 지울 수 없다. FAKE가 Track의 91.65%이므로 FAKE AP의 높은 값만 보지 않고 REAL AP·오탐을 함께 본다. Test EER의 교점 임계값은 Test 분류에 사용하지 않았다. 혼동행렬과 모든 비반올림 지표는 저장된 CSV에 있다.

### 모델별 튜닝 전후 비교

아래 네 그림은 [저장된 공동 Test CSV](../results/model_tuning/optimized_test_results.csv)의 **곡 단위** Baseline과 Optimized를 비교한다. 막대는 Optimized−Baseline의 퍼센트포인트 변화량이고, 오른쪽에는 실제 전후 값을 적었다. REAL FPR과 FAKE miss에는 각 설정의 Validation 임계값을 적용했다.

| Logistic Regression | RBF-SVM |
|---|---|
| ![LR의 Baseline과 Optimized Test 비교](../results/model_tuning/model_comparisons/logistic_regression_baseline_vs_optimized.png) | ![SVM의 Baseline과 Optimized Test 비교](../results/model_tuning/model_comparisons/rbf_svm_baseline_vs_optimized.png) |

| Log-Mel CNN | Frozen MERT + LR |
|---|---|
| ![CNN의 Baseline과 Optimized Test 비교](../results/model_tuning/model_comparisons/logmel_cnn_baseline_vs_optimized.png) | ![MERT의 Baseline과 Optimized Test 비교](../results/model_tuning/model_comparisons/frozen_mert_lr_baseline_vs_optimized.png) |

SVM은 Validation EER로 고른 설정의 Test EER과 AUC가 Baseline보다 나빠졌다. CNN은 Test EER이 낮아졌지만 REAL FPR은 높아졌다. 이 그림으로 설정을 다시 고르지 않았다.

## 7. 이 결과를 어떻게 읽을까

Optimized MERT는 Track EER **0.0667**, Balanced Accuracy **0.9363**으로 균형이 가장 좋았다. CNN은 AUC가 MERT와 거의 같고 FAKE miss가 **0.0243**으로 낮지만, REAL FPR은 **0.2000**이다. SVM은 Validation에서 선택한 설정의 Test AUC·EER가 Baseline보다 나빠졌다. 위 표의 이 차이를 그대로 보고한다.

모델·epoch·임계값은 Train/Validation에서 정하고 Test에서 바꾸지 않았다. 다만 09·13·17번 과거 실험에서 같은 Test 결과가 이미 공개된 이력이 있다. REAL Test 곡은 45개뿐이고, FMA REAL과 Echoes TTA FAKE의 출처 차이도 남는다. 따라서 작은 모델 간 차이나 다른 플랫폼에서의 성능까지 확정할 수는 없다.